In [ ]:
#!pip install torch torchvision tqdm Pillow

In [1]:
import os
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
from torch import nn
from torchvision import transforms

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
class LinearClassifier(nn.Module):
    def __init__(self, dim, num_labels=1000):
        super(LinearClassifier, self).__init__()
        self.num_labels = num_labels
        self.linear = nn.Linear(dim, num_labels)
        self.linear.weight.data.normal_(mean=0.0, std=0.01)
        self.linear.bias.data.zero_()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.linear(x)

In [3]:
class SmoothedValue:
    def __init__(self, window_size=20, fmt='{value:.4f}'):
        self.deque = []
        self.window_size = window_size
        self.total = 0.0
        self.count = 0
        self.fmt = fmt

    def update(self, value):
        self.deque.append(value)
        if len(self.deque) > self.window_size:
            self.deque.pop(0)
        self.count += 1
        self.total += value

    @property
    def global_avg(self):
        return self.total / self.count if self.count > 0 else 0

    def __str__(self):
        return self.fmt.format(value=self.global_avg)

In [13]:
class MetricLogger:
    def __init__(self, delimiter="  "):
        self.meters = {}
        self.delimiter = delimiter

    def update(self, **kwargs):
        for k, v in kwargs.items():
            if k not in self.meters:
                self.meters[k] = SmoothedValue()
            self.meters[k].update(v)

    def __getattr__(self, attr):
        if attr in self.meters:
            return self.meters[attr]
        raise AttributeError(f"'{self.__class__.__name__}' object has no attribute '{attr}'")

    def add_meter(self, name, meter=None):
        if meter is None:
            meter = SmoothedValue()
        self.meters[name] = meter

    def __str__(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append(f"{name}: {str(meter)}")
        return self.delimiter.join(loss_str)

    def log_every(self, iterable, print_freq, header=None):
        i = 0
        if header is not None:
            print(header)
        for obj in tqdm(iterable):
            yield obj
            if i % print_freq == 0:
                print(str(self))
            i += 1

In [5]:
def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

In [6]:
class SelectedImageNet(torch.utils.data.Dataset):
    def __init__(self, root_dirs, selected_classes, transform=None):
        self.root_dirs = [root_dirs] if isinstance(root_dirs, str) else root_dirs
        self.selected_classes = selected_classes
        self.transform = transform
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(list(self.selected_classes)))}
        self.samples = self._find_samples()

    def _find_samples(self):
        samples = []
        for root_dir in self.root_dirs:
            if not os.path.isdir(root_dir):
                print(f"Warning: Directory not found {root_dir}, skipping.")
                continue
            for class_name in os.listdir(root_dir):
                if class_name in self.selected_classes:
                    class_path = os.path.join(root_dir, class_name)
                    if os.path.isdir(class_path):
                        for img_name in os.listdir(class_path):
                            img_path = os.path.join(class_path, img_name)
                            if os.path.isfile(img_path) and img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                                samples.append((img_path, self.class_to_idx[class_name]))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Warning: Could not load image {img_path}. Skipping.")
            if len(self.samples) > 1:
                return self.__getitem__((idx + 1) % len(self.samples))
            else:
                raise
        if self.transform:
            image = self.transform(image)
        return image, label


In [7]:
SELECTED_CLASSES = ['n01632777', 'n01984695', 'n01728572', 'n01514859',
                    'n01614925', 'n01582220', 'n01806143', 'n01819313']


In [8]:
def train(model, linear_classifier, optimizer, loader, epoch, device, args):
    linear_classifier.train()
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter('loss', SmoothedValue(window_size=1, fmt='{value:.3f}'))
    metric_logger.add_meter('lr', SmoothedValue(window_size=1, fmt='{value:.6f}'))
    header = f'Epoch: [{epoch}]'
    for (inp, target) in metric_logger.log_every(loader, 20, header):
        inp = inp.to(device)
        target = target.to(device)
        with torch.no_grad():
            output = model(inp)
        output = linear_classifier(output)
        loss = nn.CrossEntropyLoss()(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        metric_logger.update(loss=loss.item())
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])
    print("Averaged stats:", metric_logger)
    return {k: meter.global_avg for k, meter in metric_logger.meters.items()}

In [14]:
@torch.no_grad()
def validate_network(val_loader, model, linear_classifier, device, args):
    linear_classifier.eval()
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter('acc1', SmoothedValue(window_size=1, fmt='{value:.2f}'))
    metric_logger.add_meter('loss', SmoothedValue(window_size=1, fmt='{value:.3f}'))
    header = 'Test:'
    for inp, target in metric_logger.log_every(val_loader, 20, header):
        inp = inp.to(device)
        target = target.to(device)
        output = model(inp)
        output = linear_classifier(output)
        loss = nn.CrossEntropyLoss()(output, target)
        acc1, = accuracy(output, target, topk=(1,))
        metric_logger.update(loss=loss.item())
        metric_logger.update(acc1=acc1.item())
    print('* Acc@1 {:.3f} loss {:.3f}'
          .format(metric_logger.meters['acc1'].global_avg,
                  metric_logger.meters['loss'].global_avg))
    # print('* Acc@1 {top1.global_avg:.3f} loss {losses.global_avg:.3f}'.format(
    #     top1=metric_logger.acc1, losses=metric_logger.loss))
    return {k: meter.global_avg for k, meter in metric_logger.meters.items()}

In [10]:
def train_linear(args):
    device = torch.device(args.device if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = torch.hub.load('facebookresearch/dino:main', args.arch, pretrained=True)
    embed_dim = model.embed_dim
    model.to(device)
    model.eval()

    linear_classifier = LinearClassifier(dim=embed_dim, num_labels=len(SELECTED_CLASSES)).to(device)

    train_dirs = [os.path.join(args.data_path, f"train.X{i}") for i in range(1, 5)]
    val_path = os.path.join(args.data_path, "val.X")

    transform = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    train_dataset = SelectedImageNet(train_dirs, SELECTED_CLASSES, transform)
    val_dataset = SelectedImageNet(val_path, SELECTED_CLASSES, transform)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)

    optimizer = torch.optim.SGD(
        linear_classifier.parameters(),
        args.lr * args.batch_size / 256.,
        momentum=0.9,
        weight_decay=0,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs, eta_min=0)

    best_acc = 0.0
    for epoch in range(args.epochs):
        train_stats = train(model, linear_classifier, optimizer, train_loader, epoch, device, args)
        scheduler.step()
        print(f"Train stats: {train_stats}")
        if epoch % args.val_freq == 0 or epoch == args.epochs - 1:
            test_stats = validate_network(val_loader, model, linear_classifier, device, args)
            print(f"Accuracy at epoch {epoch} of the network on the {len(val_dataset)} test images: {test_stats['acc1']:.1f}%")
            if test_stats["acc1"] > best_acc:
                torch.save({'model': linear_classifier.state_dict(), 'class_to_idx': train_dataset.class_to_idx},
                           os.path.join(args.output_dir, "best_model_for_dino_8.pth"))
            best_acc = max(best_acc, test_stats["acc1"])
            print(f'Max accuracy so far: {best_acc:.2f}%')
        torch.save({
            "epoch": epoch + 1,
            "state_dict": linear_classifier.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_acc": best_acc,
        }, os.path.join(args.output_dir, f"checkpoint_{epoch}.pth"))
    print("Training of the supervised linear classifier on frozen features completed.\n"
          "Top-1 test accuracy: {acc:.1f}".format(acc=best_acc))


In [15]:
class Args:
    arch = 'dino_vitb8'
    epochs = 4
    lr = 0.001
    batch_size = 64  # możesz zmienić w zależności od dostępnej pamięci GPU
    device = 'cuda'
    data_path = '/content/drive/MyDrive/data'
    num_workers = 10
    val_freq = 1
    output_dir = '/content/drive/MyDrive/models/dino_out'

args = Args()
Path(args.output_dir).mkdir(parents=True, exist_ok=True)
train_linear(args)

Using device: cuda


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch: [0]


  1%|          | 1/163 [00:16<45:04, 16.69s/it]

loss: 2.332  lr: 0.000250


...

Averaged stats: loss: 0.027  lr: 0.000037
Train stats: {'loss': 0.027120142608046807, 'lr': 3.66116523516817e-05}
Test:


  0%|          | 0/7 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
 14%|█▍        | 1/7 [00:07<00:45,  7.51s/it]

acc1: 98.44  loss: 0.054


100%|██████████| 7/7 [00:27<00:00,  3.91s/it]

* Acc@1 98.884 loss 0.044
Accuracy at epoch 3 of the network on the 400 test images: 98.9%
Max accuracy so far: 98.88%
Training of the supervised linear classifier on frozen features completed.
Top-1 test accuracy: 98.9
